In [25]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def jalankan_pipeline():
    print("1. Membaca dataset mentah...")
    df_trx = pd.read_csv("data/transaksi.csv")
    df_cust = pd.read_csv("data/pelanggan.csv")

    print("2. Membersihkan data transaksi & pelanggan...")
    # Bersihkan transaksi
    df_trx = df_trx.drop_duplicates(subset=["id_transaksi"]).copy()
    df_trx["tanggal"] = pd.to_datetime(df_trx["tanggal"], format="mixed")
    df_trx["kategori"] = df_trx["kategori"].fillna("Lainnya")
    df_trx["metode_pembayaran"] = df_trx["metode_pembayaran"].fillna("Lainnya").str.strip().str.upper()
    df_trx["harga_satuan"] = (
        df_trx["harga_satuan"].astype(str)
        .str.replace("Rp", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.strip()
        .astype(int)
    )

    # Bersihkan pelanggan
    df_cust["kota"] = df_cust["kota"].fillna("Tidak Diketahui").str.strip().str.title()

    print("3. Menggabungkan tabel (Left Join)...")
    df_gabung = pd.merge(df_trx, df_cust, on="id_pelanggan", how="left")

    # Ambil hanya transaksi yang sukses selesai
    df_valid = df_gabung[df_gabung["status_pengiriman"] == "Selesai"].copy()

    print("4. Menghitung ringkasan bisnis...")
    ringkasan_kota = (
        df_valid.groupby("kota")
        .agg(
            total_transaksi=("id_transaksi", "count"),
            total_omzet=("total_harga", "sum"),
            total_profit=("profit", "sum")
        )
        .sort_values("total_omzet", ascending=False)
        .reset_index()
    )

    ringkasan_produk = (
        df_valid.groupby(["kategori", "nama_produk"])
        .agg(
            unit_terjual=("qty", "sum"),
            total_omzet=("total_harga", "sum"),
            total_profit=("profit", "sum")
        )
        .sort_values("total_profit", ascending=False)
        .reset_index()
    )
    
    # Ambil hanya transaksi yang sukses selesai
    df_platinum = df_gabung[df_gabung["segmen"] == "Platinum"].copy()
    ringkasan_metode_pembayaran = (
        df_platinum.groupby("metode_pembayaran")
        .agg(
            total_transaksi=("id_transaksi", "nunique")
        )
        .sort_values("total_transaksi", ascending=False)
        .reset_index()
    )
    
    ringkasan_kategori = (
        df_valid.groupby("kategori")
        .agg(
            total_profit_kategori=("profit", "sum")
        )
        .sort_values("total_profit_kategori", ascending=False)
        .reset_index()
    )
    
    df_invalid = df_gabung[df_gabung["status_pengiriman"] == "Dibatalkan"].copy()
    print("Total kerugiian", np.sum(df_invalid["total_harga"]))

    ringkasan_kota_dibatalkan = (
        df_invalid.groupby("kota")
        .agg(
            total_transaksi=("id_transaksi", "nunique"),  # Gunakan nunique agar tidak double count
            potensi_omzet_hilang=("total_harga", "sum"),
            potensi_profit_hilang=("profit", "sum")
        )
        .sort_values("potensi_omzet_hilang", ascending=False)
        .reset_index()
    )

    ringkasan_produk_dibatalkan = (
        df_invalid.groupby(["kategori", "nama_produk"])
        .agg(
            unit_batal=("qty", "sum"),  # Penamaan lebih akurat daripada unit_terjual
            potensi_omzet_hilang=("total_harga", "sum"),
            potensi_profit_hilang=("profit", "sum")
        )
        .sort_values("potensi_profit_hilang", ascending=False)
        .reset_index()
    )

    print("5. Membuat visualisasi...")
    os.makedirs("reports", exist_ok=True)
    plt.figure(figsize=(9, 5))
    sns.barplot(data=ringkasan_kota, x="total_omzet", y="kota", palette="Blues_r")
    plt.title("Total Omzet Berdasarkan Kota Pelanggan", fontweight="bold")
    plt.xlabel("Total Omzet (IDR)")
    plt.ylabel("Kota")
    plt.tight_layout()
    plt.savefig("reports/omzet_per_kota.png", dpi=150)
    plt.close()
    
    os.makedirs("reports", exist_ok=True)
    plt.figure(figsize=(9, 5))
    sns.barplot(data=ringkasan_metode_pembayaran, x="total_transaksi", y="metode_pembayaran", hue="metode_pembayaran", palette="Blues_r")
    plt.title("Total transaksi Berdasarkan metode pembayaran", fontweight="bold")
    plt.xlabel("Total Transaksi")
    plt.ylabel("Metode Pembayaran")
    plt.tight_layout()
    plt.savefig("reports/metode_pembayaran.png", dpi=150)
    plt.close()
    
    os.makedirs("reports", exist_ok=True)
    plt.figure(figsize=(9, 5))
    sns.barplot(data=ringkasan_kategori, x="total_profit_kategori", y="kategori", hue="kategori", palette="Blues_r")
    plt.title("Total profit Berdasarkan kateogori", fontweight="bold")
    plt.xlabel("Total profit")
    plt.ylabel("kategori")
    plt.tight_layout()
    plt.savefig("reports/kategori.png", dpi=150)
    plt.close()
    
    os.makedirs("reports", exist_ok=True)
    plt.figure(figsize=(9, 5))
    
    # Tambahkan hue & legend=False untuk menghindari FutureWarning Seaborn
    ax = sns.barplot(
        data=ringkasan_kota_dibatalkan,
        x="potensi_omzet_hilang",  # Sesuaikan jika nama kolom diubah
        y="kota",
        hue="kota",
        palette="Reds_r",
        legend=False
    )
    
    # Format sumbu X agar tidak muncul notasi ilmiah (1e7)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
    
    plt.title("Total Potensi Kerugian Omzet Berdasarkan Kota Pelanggan", fontweight="bold")
    plt.xlabel("Total Kerugian Omzet (IDR)")
    plt.ylabel("Kota")
    plt.tight_layout()
    plt.savefig("reports/omzet_dibatalkan_per_kota.png", dpi=150)
    plt.close()


    print("6. Mengekspor laporan ke Excel...")
    output_excel = "reports/laporan_akhir_2026.xlsx"
    with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
        ringkasan_kota.to_excel(writer, sheet_name="Performa_Kota", index=False)
        ringkasan_produk.to_excel(writer, sheet_name="Performa_Produk", index=False)
        df_valid.to_excel(writer, sheet_name="Data_Bersih", index=False)

    print(f"\nSELESAI! Cek hasil laporan di:")
    print(f"- File Excel : {output_excel}")
    print(f"- File Grafik: reports/omzet_per_kota.png")
    
    print("6. Mengekspor laporan ke Excel...")
    output_excel = "reports/laporan_dibatalkan_2026.xlsx"
    with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
        ringkasan_kota_dibatalkan.to_excel(writer, sheet_name="Kerugian_Kota", index=False)
        ringkasan_produk_dibatalkan.to_excel(writer, sheet_name="Kerugian_Produk", index=False)
        df_invalid.to_excel(writer, sheet_name="Data_Dibatalkan", index=False)

    print(f"\nSELESAI! Cek hasil laporan di:")
    print(f"- File Excel : {output_excel}")
    print(f"- File Grafik: reports/omzet_dibatalkan_per_kota.png")


jalankan_pipeline()


1. Membaca dataset mentah...
2. Membersihkan data transaksi & pelanggan...
3. Menggabungkan tabel (Left Join)...
4. Menghitung ringkasan bisnis...
Total kerugiian 218964500
5. Membuat visualisasi...


/tmp/ipykernel_111721/2563219069.py:106: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=ringkasan_kota, x="total_omzet", y="kota", palette="Blues_r")


6. Mengekspor laporan ke Excel...

SELESAI! Cek hasil laporan di:
- File Excel : reports/laporan_akhir_2026.xlsx
- File Grafik: reports/omzet_per_kota.png
6. Mengekspor laporan ke Excel...

SELESAI! Cek hasil laporan di:
- File Excel : reports/laporan_dibatalkan_2026.xlsx
- File Grafik: reports/omzet_dibatalkan_per_kota.png
